# Original MRA on Kaggle (branch `mra_Og` of the RaCA fork): train and evaluate the baseline

**Before running**
1. *Add Input* → attach the **scenes** dataset (output of the step-2 subset notebook; it contains `kitti_subset/selection.json`).
2. Optional: attach a dataset holding `ckpts/warm_start.pth` (e.g. `mra-before-run`) so exactly the same starting weights are reused; without it they are rebuilt from SGN-T (~1 min).
3. *Settings*: Accelerator **GPU T4**, Internet **On**.
4. Set `RUN` below (`mra_before`) and click *Run All*. When `Epoch [1][50/400]` lines appear, stop the session and use *Save Version → Save & Run All (Commit)* so the ~5 h run continues in the background.

The run starts from the SGN-T warm start shared with the RaCA runs, with seed 0. MRA's own initialisation does not fit a T4: a random occupancy head seeds about half the voxel grid and the first step needs > 13 GB (see `kaggle/README.md`). `--deterministic` is off by default (`DETERMINISTIC=1` in `train.sh`).

Everything this notebook does is in the repository's `kaggle/` folder; see `kaggle/README.md`.

In [ ]:
# ---- Settings ----
RUN      = "mra_before"     # which kaggle/configs/<RUN>.py to train
REPO_URL = "https://github.com/fafnirLore/RaCA.git"
BRANCH   = "mra_Og"              # original MRA + kaggle/ folder, no RaCA code
EXTRA    = ""                 # extra train.py args, e.g. "--cfg-options runner.max_epochs=3"
import os
REPO = f"/kaggle/working/RaCA_Og"
os.environ.update(dict(RUN=RUN, REPO=REPO, REPO_URL=REPO_URL, BRANCH=BRANCH, EXTRA=EXTRA))
print("run:", RUN, "| repo:", REPO)

In [ ]:
%%bash
# 1. Code + software (~15 min the first time; skipped when already installed in this session)
nvidia-smi --query-gpu=name,memory.total --format=csv
if [ -d "$REPO/.git" ]; then cd "$REPO" && git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git reset -q --hard "origin/$BRANCH"
else git clone -q -b "$BRANCH" "$REPO_URL" "$REPO" && cd "$REPO"; fi
git log --oneline -1    # must show the commit you expect; an existing clone is reset to the branch head on GitHub
bash kaggle/install.sh && bash kaggle/get_weights.sh sgn

In [ ]:
# 2. Data: find the saved subset and rebuild MRA's file layout at /tmp/kitti_full/ (~3 min)
import glob, json, subprocess, os
hits = glob.glob("/kaggle/input/**/kitti_subset/selection.json", recursive=True)
assert hits, "scenes dataset not attached (see the top of this notebook)"
SRC = os.path.dirname(hits[0]) + "/"
s = json.load(open(hits[0]))["summary"]
print("subset:", SRC, f"| train frames {s['train']['frames']} | val frames {s['val']['frames']}")
if not os.path.exists("/tmp/kitti_full/dataset/labels"):
    subprocess.run(["python", f"{REPO}/kaggle/expand_subset.py", SRC, "/tmp/kitti_full/"], check=True)

In [ ]:
# 3. Starting weights: reuse an attached warm_start.pth, otherwise build it from SGN-T (~1 min).
#    Shared with the RaCA runs; MRA's own initialisation does not fit a T4 (see the top of this notebook).
import glob, subprocess, os
subprocess.run(["bash", f"{REPO}/kaggle/ensure_warm_start.sh"], check=True)
assert os.path.getsize(f"{REPO}/ckpts/warm_start.pth") > 0
BEFORE = glob.glob("/kaggle/input/**/results_trained.json", recursive=True)
os.environ["BEFORE_JSON"] = BEFORE[0] if BEFORE else ""
print("earlier results for comparison:", BEFORE[0] if BEFORE else "none attached")


In [ ]:
%%bash
# 4. Train (~40-55 min per epoch on a T4). Lines like `Epoch [1][50/400] ... loss: ...` show progress.
cd "$REPO" && bash kaggle/train.sh kaggle/configs/$RUN.py $RUN $EXTRA

In [ ]:
%%bash
# 5. Evaluate the trained model on the 150 validation frames (~5 min)
cd "$REPO" && bash kaggle/eval.sh kaggle/configs/$RUN.py /kaggle/working/runs/$RUN/latest.pth /kaggle/working/runs/$RUN/results_trained.json "$RUN: trained" 

In [ ]:
%%bash
# 6. What is kept as the notebook output (checkpoint, logs, results); the dataset copy in /tmp is not saved
du -sh /kaggle/working/runs/$RUN; ls /kaggle/working/runs/$RUN